In [ ]:
# Install deps
!pip install -q pandas scikit-learn joblib tqdm datasets

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
sys.path.insert(0, '.')
print("Repo ready")

In [ ]:
# Download data and model assets
!python scripts/download_assets.py

In [ ]:
# Run RAID benchmark evaluation
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from tool.feature_extractor import extract_features
import joblib

# Load stylometric model
model = joblib.load("models/detector_all.joblib")

# Sample train set (RAID has no test split)
# Load dataset from Hugging Face
from datasets import load_dataset
raid = load_dataset("liamdugan/raid")

test_df = raid['train'].to_pandas()
sample_size = min(5000, len(test_df))
test_sample = test_df.sample(n=sample_size, random_state=42)
print(f"Evaluating on {len(test_sample)} texts")

# Extract features and predict
features = []
labels = []
for i, row in test_sample.iterrows():
    text = row['text']
    label = row['label']  # 0=human, 1=AI
    if not isinstance(text, str) or len(text.strip()) < 20:
        continue
    
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    
    X = np.array([feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                        'connector_density', 'hedge_density', 'mean_sent_len',
                                        'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']])
    features.append(X)
    labels.append(label)

features = np.array(features)
labels = np.array(labels)

# Predict
probs = model.predict_proba(features)[:, 1]
auc = roc_auc_score(labels, probs)
print(f"RAID AUC: {auc:.4f}")

# Save results
results = pd.DataFrame([{'benchmark': 'RAID', 'auc': auc, 'n_samples': len(labels)}])
results.to_csv('/kaggle/working/raid_results.csv', index=False)
print("Results saved")